In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

In [3]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

In [7]:
data = []

base_url = "https://www.alibaba.com/trade/search?SearchText=Auto+Accessories&page="

for page in range(1, 6):
    print("\nOpening Page:", page)
    driver.get(base_url + str(page))
    try:
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    except:
        print("Page load timeout")
        continue
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(3)
    time.sleep(8)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = soup.find_all("a")
    print("Links found:", len(links))

    if len(links) < 50:
        print("Low content page detected, retrying...")
        time.sleep(5)
        soup = BeautifulSoup(driver.page_source, "html.parser")

    products = soup.select("a[href*='product'], a[href*='offer'], a[href*='item']")
    print("Products found:", len(products))
    seen = set()

    for p in products:
        title = p.get_text(strip=True)
        if len(title) < 10:
            continue
        if title in seen:
            continue
        seen.add(title)
        link = p.get("href", "N/A")
        data.append([title, link])


Opening Page: 1
Links found: 655
Products found: 437

Opening Page: 2
Links found: 647
Products found: 434

Opening Page: 3
Links found: 624
Products found: 408

Opening Page: 4
Links found: 633
Products found: 418

Opening Page: 5
Links found: 628
Products found: 418


In [8]:
df = pd.DataFrame(data, columns=["Product Title", "Product Link"])
df.drop_duplicates(inplace=True)
df.to_csv("alibaba_products.csv", index=False, encoding="utf-8")